# BSM L04D (wersja 2026/2027) — MFA w aplikacji mobilnej: TOTP, limity prób, replay i fallback

Notebook służy do wypełnienia i wysłania odpowiedzi. Kod implementujesz lokalnie w Android Studio / Kotlin.

## Repozytorium kursu
Otwórz repozytorium GitHub:
- https://github.com/duszekjk/mobile-systems-security.git

Jeśli klonujesz repo pierwszy raz:
- `git clone https://github.com/duszekjk/mobile-systems-security.git`
- `cd mobile-systems-security`

## Aktualizacja tylko startera L04D (bez ryzyka dla poprzednich labów)
Rekomendowane w tym kursie: aktualizuj tylko foldery startera tego labu.

1) Pobierz najnowszy stan z GitHuba (nie zmienia jeszcze plików):
- `git fetch origin`

2) Nadpisz tylko starter L04D i arkusz:
- `git checkout origin/main -- student/apps/lesson_d_app`
- `git checkout origin/main -- student/labs/BSM_L04_MFA_TOTP_Fallback_v2.ipynb`

3) Zapisz to jako commit:
- `git add -A && git commit -m "L04D: update starter"`

Uwaga: to nadpisuje lokalne zmiany w `student/apps/lesson_d_app`. Jeśli chcesz zachować swoje zmiany w tym folderze, pracuj na osobnym branchu.

## Android Studio (klik po kliku)
1. Android Studio → **File → Open...**
2. Wybierz folder: `student/apps/lesson_d_app`
3. **Trust Project** (jeśli zapyta)
4. Poczekaj na Gradle Sync.
5. Jeśli trzeba ustawić JDK 17:
   - **Settings → Build, Execution, Deployment → Build Tools → Gradle → Gradle JDK → 17**

## CLI `./gradlew` też wymaga JDK 17
Jeśli uruchamiasz testy z terminala, sprawdź:
- `cd student/apps/lesson_d_app && ./gradlew -version`

Jeśli to nie Java 17 (np. widzisz 25), ustaw na macOS:
- `export JAVA_HOME=$(/usr/libexec/java_home -v 17)`
- `export PATH="$JAVA_HOME/bin:$PATH"`

## Zasada pracy
- Każde zadanie wysyłasz osobno: uruchom komórkę formularza + komórkę `zapisz_i_wyslij(...)`.
- Jeśli zadanie wymaga logu lub wyniku testu: wklej tylko wymagany fragment.


In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


In [ ]:
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("\n", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    wyslij_odpowiedz(task_id, final_answer)


# D01 — Klasyfikacja metod MFA

Uzupełnij formularz etykietami: `REPLAY_RESISTANT`, `PHISHING_RESISTANT`, `CHANNEL_WEAK`, `SERVER_SECRET_EXPOSED`.

## Jak uzyskać odpowiedź (co wysłać)
- Wybierz wartości w formularzu i wyślij `final_answer`.


In [ ]:
#@title D01 — Formularz odpowiedzi
sms_role = "" #@param ["", "REPLAY_RESISTANT", "PHISHING_RESISTANT", "CHANNEL_WEAK", "SERVER_SECRET_EXPOSED"]
totp_role = "" #@param ["", "REPLAY_RESISTANT", "PHISHING_RESISTANT", "CHANNEL_WEAK", "SERVER_SECRET_EXPOSED"]
skey_role = "" #@param ["", "REPLAY_RESISTANT", "PHISHING_RESISTANT", "CHANNEL_WEAK", "SERVER_SECRET_EXPOSED"]
u2f_role = "" #@param ["", "REPLAY_RESISTANT", "PHISHING_RESISTANT", "CHANNEL_WEAK", "SERVER_SECRET_EXPOSED"]

final_answer = prepare_answer(
    "sms=" + sms_role,
    "totp=" + totp_role,
    "skey=" + skey_role,
    "u2f=" + u2f_role,
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D01", final_answer)


# D02 — Provisioning sekretu TOTP

W tym zadaniu oceniasz stan startera przed naprawą. Przejdź po kodzie od `SecureSessionStore.ensureProvisioned()` do ekranu MFA.

Szukasz:
1) gdzie jest sekret,
2) czy sekret/`otpauth://...` trafia do logów,
3) gdzie jest provisioning.

## Jak uzyskać odpowiedź (co wysłać)
- Otwórz wskazane klasy i uzupełnij formularz.
- Wyślij `final_answer`.


In [ ]:
#@title D02 — Formularz odpowiedzi
provisioning_place = "" #@param {type:"string"}
totp_secret_store = "" #@param {type:"string"}
secret_in_logs = "" #@param ["", "YES", "NO"]
issuer_or_uri_prefix = "" #@param {type:"string"}

final_answer = prepare_answer(
    "place=" + short_text(provisioning_place),
    "store=" + short_text(totp_secret_store),
    "log=" + secret_in_logs,
    "prefix=" + short_text(issuer_or_uri_prefix, limit=16),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D02", final_answer)


# D03 — Implementacja TOTP w Kotlinie

Edytujesz:
- `student/apps/lesson_d_app/app/src/main/java/com/example/secretlab/mfa/TotpEngine.kt`

Uzupełnij `TODO(D03-...)`.

## Jak uzyskać odpowiedź (co wysłać)
1. Uruchom test: `TotpEngineStudentTest`.
2. Wpisz do formularza:
   - nazwę klasy,
   - nazwę testu referencyjnego (`matchesReferenceVectorAtKnownTimestamp`),
   - `YES/NO` czy pakiet testów przeszedł,
   - `digits`.
3. Wyślij `final_answer`.


In [ ]:
#@title D03 — Formularz odpowiedzi
totp_engine_class = "" #@param {type:"string"}
totp_reference_test = "" #@param {type:"string"}
totp_tests_passed = "" #@param ["", "YES", "NO"]
otp_digits = "" #@param {type:"string"}

final_answer = prepare_answer(
    "engine=" + short_text(totp_engine_class),
    "test=" + short_text(totp_reference_test),
    "pass=" + totp_tests_passed,
    "digits=" + short_text(otp_digits, limit=8),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D03", final_answer)


# D04 — Okno czasowe i drift

Sprawdzasz, jak `window` wpływa na akceptację kodów z sąsiednich slotów.

Edytujesz/oglądasz:
- `TotpEngine.kt`
- `MainActivity.kt` (tworzenie `TotpEngine(window = ...)`)

## Jak uzyskać odpowiedź (co wysłać)
1. Uruchom test: `acceptsCodeFromAdjacentTimeStepWhenWindowIsOne`.
2. Uzupełnij formularz (window, adj, policy) i wyślij `final_answer`.


In [ ]:
#@title D04 — Formularz odpowiedzi
totp_window_size = "" #@param {type:"string"}
adjacent_slot_accepted = "" #@param ["", "YES", "NO"]
window_policy = "" #@param ["", "strict", "lenient"]

final_answer = prepare_answer(
    "window=" + short_text(totp_window_size, limit=8),
    "adj=" + adjacent_slot_accepted,
    "policy=" + window_policy,
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D04", final_answer)


# D05 — Limity prób i blokada

Edytujesz:
- `student/apps/lesson_d_app/app/src/main/java/com/example/secretlab/mfa/AttemptLimiter.kt`

Uzupełnij `TODO(D05-...)`.

## Jak uzyskać odpowiedź (co wysłać)
1. Uruchom test: `AttemptLimiterStudentTest.blocksAfterConfiguredNumberOfFailures`.
2. Uzupełnij formularz i wyślij `final_answer`.


In [ ]:
#@title D05 — Formularz odpowiedzi
attempt_limiter_class = "" #@param {type:"string"}
max_bad_attempts = "" #@param {type:"string"}
cooldown_seconds = "" #@param {type:"string"}
limiter_test_passed = "" #@param ["", "YES", "NO"]

final_answer = prepare_answer(
    "limiter=" + short_text(attempt_limiter_class),
    "max=" + short_text(max_bad_attempts, limit=8),
    "cool=" + short_text(cooldown_seconds, limit=8),
    "pass=" + limiter_test_passed,
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D05", final_answer)


# D06 — Replay i związanie kodu z próbą logowania

Edytujesz:
- `student/apps/lesson_d_app/app/src/main/java/com/example/secretlab/mfa/MfaLoginGateway.kt`

Cel: wiązać weryfikację z `challengeId` i blokować replay zaakceptowanego kroku czasu.

## Jak uzyskać odpowiedź (co wysłać)
1. Uruchom testy:
   - `rejectsOtpForDifferentChallengeId`
   - `rejectsReplayOfCodeFromAlreadyAcceptedTimeStep`
2. Uzupełnij formularz i wyślij `final_answer`.


In [ ]:
#@title D06 — Formularz odpowiedzi
session_binding_artifact = "" #@param {type:"string"}
otp_reusable = "" #@param ["", "YES", "NO"]
binding_assessment = "" #@param ["", "bound", "unbound"]

final_answer = prepare_answer(
    "bind=" + short_text(session_binding_artifact),
    "reuse=" + otp_reusable,
    "mode=" + binding_assessment,
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D06", final_answer)


# D07 — Słaby fallback obchodzi mocne MFA

Oceń ścieżkę fallback i czy omija TOTP.

Edytujesz/oglądasz:
- `MainActivity.kt` (ekran fallback)
- `MfaLoginGateway.fallbackRecover(...)`

## Jak uzyskać odpowiedź (co wysłać)
- Uzupełnij formularz i wyślij `final_answer`.


In [ ]:
#@title D07 — Formularz odpowiedzi
fallback_mechanism = "" #@param {type:"string"}
fallback_bypasses_totp = "" #@param ["", "YES", "NO"]
fallback_assessment = "" #@param ["", "weaker_than_main", "comparable_to_main"]

final_answer = prepare_answer(
    "fallback=" + short_text(fallback_mechanism),
    "bypass=" + fallback_bypasses_totp,
    "risk=" + fallback_assessment,
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D07", final_answer)


# D08 — Ocena końcowa architektury logowania

To syntetyczna ocena: co system realnie osiąga po poprawkach, a co nadal jest najsłabsze.

## Jak uzyskać odpowiedź (co wysłać)
- Uzupełnij formularz i wyślij `final_answer`.


In [ ]:
#@title D08 — Formularz odpowiedzi
replay_resistant = "" #@param ["", "YES", "NO"]
phishing_resistant = "" #@param ["", "YES", "NO"]
weakest_element = "" #@param ["", "fallback", "provisioning", "device_compromise", "logging", "time_window"]
next_arch_step = "" #@param {type:"string"}

final_answer = prepare_answer(
    "replay=" + replay_resistant,
    "phish=" + phishing_resistant,
    "weak=" + weakest_element,
    "next=" + short_text(next_arch_step, limit=80),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D08", final_answer)
